
# TP 2 — Optimisation manuelle (descente de gradient en PyTorch)

**Objectifs de la séance.**
- Terminer l'ajustement de la régression linéaire démarrée en séance 1 (descente de gradient *full batch*).
- Étendre le code à la descente de gradient mini-batch et stochastique, et observer l'effet sur la variance du gradient et la vitesse de convergence.
- Observer l'effet du *learning rate* (trop grand, trop petit, bon compromis).
- Découvrir et utiliser `torch.optim`, en particulier l'optimiseur Adam.

Pas de `nn.Module` ni de `Trainer` aujourd'hui : on reste en PyTorch « à la main », l'abstraction arrive en séance 3.

In [ ]:

import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)


## Point de départ : reprise de la séance 1

Voici le code exact sur lequel s'est terminée la séance 1 (ne modifiez pas la génération du jeu de données : vous devez retrouver les mêmes valeurs cibles $w^\star=-3$, $b^\star=1.5$). Pas besoin d'avoir gardé le notebook de la séance 1 ouvert : tout est redonné ci-dessous.

In [ ]:

X = torch.rand(100, 1)
# w* = -3, b* = 1.5
y = -3.0 * X + 1.5 + 0.4 * torch.randn(X.size())

def mse(X, y, w, b):
    y_pred = w * X + b
    return torch.mean((y_pred - y) ** 2)

plt.scatter(X.numpy(), y.numpy())
plt.xlabel("X"); plt.ylabel("y")


## Partie 1 — Terminer la régression *full batch*

**Question 1.1.** Écrivez la boucle de descente de gradient *full batch* : initialisez `w` et `b` à 0 (tenseurs scalaires `requires_grad=True`), puis itérez pendant `n_iter = 1000` pas avec `stepsize = 0.1`. Stockez la valeur de la perte à chaque itération dans une liste `losses_full` (elle servira aux comparaisons de la partie 2).

In [ ]:

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

stepsize = 0.1
n_iter = 1000
losses_full = []

for i in range(n_iter):
    # TODO : calculer la perte, la stocker dans losses_full, déclencher le calcul du gradient,
    #        mettre à jour w et b (dans un bloc torch.no_grad()), puis remettre les gradients à zéro
    pass


**Question 1.2.** Affichez `losses_full` en fonction du nombre d'itérations, et affichez les valeurs finales de `w` et `b`. Sont-elles proches de $w^\star=-3$ et $b^\star=1.5$ ?

In [ ]:

plt.plot(losses_full)
plt.xlabel("itération"); plt.ylabel("MSE")
plt.title("Descente de gradient full batch")
plt.show()

print("w appris :", w.item(), " (attendu : -3)")
print("b appris :", b.item(), " (attendu : 1.5)")


## Partie 2 — Full batch vs mini-batch vs stochastique

Jusqu'ici, chaque pas de gradient utilisait l'intégralité des 100 exemples (« full batch »). On va maintenant comparer trois façons de calculer le gradient :

- **full batch** : le gradient est calculé sur tout le jeu de données à chaque pas (ce que vous venez de faire) ;
- **mini-batch** : le gradient est calculé sur un petit sous-ensemble tiré aléatoirement à chaque pas (par exemple 10 exemples) ;
- **stochastique** : cas extrême du mini-batch, avec un seul exemple à la fois.

Pour comparer sur un pied d'égalité, on raisonne en **époques** : une époque = un passage complet sur les 100 exemples, qu'il soit fait en 1 pas (full batch), 10 pas (mini-batch de taille 10) ou 100 pas (stochastique).

**Question 2.1.** Écrivez une fonction `gradient_descent(X, y, batch_size, n_epochs, stepsize)` qui réinitialise `w` et `b` à 0, puis effectue `n_epochs` époques de descente de gradient avec la taille de batch demandée (à chaque époque, mélangez les indices avec `torch.randperm(len(X))`, puis parcourez les mini-lots consécutifs de cette permutation). La fonction doit renvoyer `w`, `b`, et la liste des pertes calculées sur **l'ensemble** du jeu de données à la fin de chaque époque (pas seulement sur le dernier mini-lot), pour pouvoir comparer les courbes entre elles.

In [ ]:

def gradient_descent(X, y, batch_size, n_epochs, stepsize):
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    losses = []

    for epoch in range(n_epochs):
        # TODO : mélanger les indices avec torch.randperm(len(X))

        # TODO : parcourir les mini-lots consécutifs de taille batch_size et faire, pour chacun,
        #        un pas de descente de gradient (perte, backward, mise à jour, remise à zéro)

        with torch.no_grad():
            pass  # TODO : calculer la perte mse(X, y, w, b) sur TOUT le dataset et l'ajouter à losses

    return w, b, losses


**Question 2.2.** Appelez `gradient_descent` avec `batch_size` égal à 100 (full batch), 10 (mini-batch) et 1 (stochastique), pour `n_epochs = 30` et `stepsize = 0.1`. Tracez les trois courbes de perte (par époque) sur un même graphique.

In [ ]:

# TODO : appeler gradient_descent avec batch_size=100, 10 puis 1 (mêmes n_epochs et stepsize)
#        et tracer les trois courbes de perte sur un même graphique (une légende par courbe)


**Questions.**
- Quelle courbe est la plus « bruitée » d'une époque à l'autre ? Pourquoi (pensez à la variance de l'estimateur du gradient selon la taille du batch) ?
- Laquelle converge le plus vite *en nombre d'époques* ? Laquelle est la plus coûteuse *en nombre de mises à jour de `w`/`b`* ?


_Votre réponse ici._


## Partie 3 — Effet du learning rate

**Question 3.1.** En reprenant la version mini-batch (`batch_size=10`) de la partie 2, comparez plusieurs valeurs de `stepsize` : une très petite (ex. 0.001), une « raisonnable » (ex. 0.1) et une trop grande (ex. 2.0 ou plus). Tracez les courbes de perte correspondantes sur un même graphique (`n_epochs=30`).

In [ ]:

# TODO : comparer stepsize=0.001, 0.1 et 2.0 (ou plus) avec batch_size=10, n_epochs=30
#        et tracer les courbes de perte correspondantes


**Questions.** Que se passe-t-il avec un learning rate trop grand ? Trop petit ? Ce compromis sera au cœur des séances suivantes (Adam, entre autres, essaie d'automatiser ce réglage).


_Votre réponse ici._


## Partie 4 — `torch.optim` et Adam

Jusqu'ici, la mise à jour `w -= stepsize * w.grad` était écrite à la main. PyTorch fournit un module `torch.optim` qui encapsule cette logique : on lui donne les paramètres à optimiser, et il s'occupe de la mise à jour (et gère plus de subtilités qu'une simple descente de gradient, comme on va le voir avec Adam).

**Question 4.1.** Réécrivez la boucle *full batch* de la partie 1, mais en remplaçant la mise à jour manuelle par un `torch.optim.SGD([w, b], lr=0.1)` : à chaque itération, appelez `optimizer.zero_grad()`, `loss.backward()`, puis `optimizer.step()` (plus besoin de `torch.no_grad()` ni de `.grad.zero_()` manuels, l'optimiseur s'en charge). Vérifiez que vous retrouvez (à peu de choses près) les mêmes courbes de perte qu'en partie 1.

In [ ]:

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.1)
losses_sgd = []

for i in range(1000):
    # TODO : optimizer.zero_grad(), calcul de la perte, loss.backward(), optimizer.step()
    #        et stockage de la perte dans losses_sgd
    pass

plt.plot(losses_full, label="mise à jour manuelle")
plt.plot(losses_sgd, label="torch.optim.SGD")
plt.legend(); plt.xlabel("itération"); plt.ylabel("MSE")


Adam est un optimiseur adaptatif : il maintient, pour chaque paramètre, une moyenne mobile du gradient (une composante de type *momentum*) et une moyenne mobile du carré du gradient (pour adapter le learning rate paramètre par paramètre), avec une correction de biais en début d'entraînement. On ne le réimplémente pas : on l'utilise tel quel.

**Question 4.2.** Reprenez la boucle précédente en remplaçant `torch.optim.SGD` par `torch.optim.Adam([w, b], lr=0.1)`. Comparez la courbe de perte obtenue à celle obtenue avec SGD, pour le même learning rate.

In [ ]:

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.Adam([w, b], lr=0.1)
losses_adam = []

# TODO : même boucle qu'au-dessus (1000 itérations) mais avec l'optimiseur Adam,
#        en stockant la perte dans losses_adam

plt.plot(losses_sgd, label="SGD (lr=0.1)")
plt.plot(losses_adam, label="Adam (lr=0.1)")
plt.legend(); plt.xlabel("itération"); plt.ylabel("MSE")


**Question.** À learning rate identique, Adam est-il plus rapide, plus lent, plus stable que SGD sur ce problème ? Le constat serait-il forcément le même sur un problème plus complexe (paysage de la perte moins « gentil » qu'une régression linéaire) ?


_Votre réponse ici._


## Bilan

Vous savez maintenant écrire une boucle d'optimisation complète en PyTorch, comparer full batch / mini-batch / stochastique, régler un learning rate, et utiliser `torch.optim`. La séance 3 introduit `nn.Module` et construit progressivement un `Trainer` réutilisable pour le reste du cours.